# Phase 4 -- Transaction / Rule Feature Representation

Alert Intelligence Engine -- master plan Phase 4 (section 20): "Behavioural representation + rule context." Gate: feature audit passes.

Covers `Rule` (Transaction Rule Alerts) -- the transaction-monitoring / customer-behaviour pipeline, separate from the entity/watchlist pipeline built in Phase 3. Per the feasibility report: Rule alerts "primarily concern transaction and customer behavioural abnormality," not name matching.

**No model training happens in this phase** -- that's Phase 6. This builds representations only.

**The most important thing to know about this phase going in:** the current sample has **no clean structured transaction amount field** (Phase 1 finding). Amounts appear only inside `Comment`, which is itself a leakage field (post-review text, created after a human reviewed the alert). Using it -- for anything, including amount extraction -- would smuggle leakage into a feature under a different name. This module has no code path that reads `Comment`, verified by a source-scanning test, not just a docstring promise. Transaction amount stays excluded until the client supplies a structured source field.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Load Phase 2 normalized Rule data

In [2]:
from pipelines.normalization.pipeline import run_phase2_pipeline

normalized_sheets, phase2_report = run_phase2_pipeline(REPO_ROOT, persist=False)
assert phase2_report["overall_status"] == "PASS"

rule_df = normalized_sheets["Rule"]
print(f"Rule: {len(rule_df)} rows, {len(rule_df.columns)} columns")

Rule: 2244 rows, 56 columns


## 2. What's available vs. excluded, and why

Grounded in the actual cardinalities/content checked directly against the workbook, not assumed.

In [3]:
from features.transaction_features import CONTEXT_CATEGORICAL_COLUMNS, NATIONALITY_COLUMN, BENEFICIARY_NAME_COLUMN

print("Included context categorical features:")
for col in CONTEXT_CATEGORICAL_COLUMNS:
    print(f"  {col}: {rule_df[col].nunique()} distinct values")

print(f"\nNationality feature: {NATIONALITY_COLUMN}")
print(f"Beneficiary name representation: {BENEFICIARY_NAME_COLUMN} "
      f"({rule_df['Beneficiary Name_missing'].mean()*100:.1f}% missing)")

print("\nDeliberately excluded, and why:")
print("  - transaction amount: no structured field; only in the leakage-typed Comment field")
print(f"  - Customer Currency: constant in this sample "
      f"({rule_df['Customer Currency'].nunique()} distinct value) -- zero signal")
print("  - Purpose Code: redundant with Purpose (same cardinality/coverage)")

Included context categorical features:
  Transaction Type Code: 6 distinct values
  Branch Description: 39 distinct values
  Currency Name: 31 distinct values
  Beneficiary Relationship: 50 distinct values
  Purpose: 23 distinct values
  Rule Name: 18 distinct values

Nationality feature: Customer Nationality (Normalized)
Beneficiary name representation: Beneficiary Name (Normalized) (22.9% missing)

Deliberately excluded, and why:
  - transaction amount: no structured field; only in the leakage-typed Comment field
  - Customer Currency: constant in this sample (1 distinct value) -- zero signal
  - Purpose Code: redundant with Purpose (same cardinality/coverage)


## 3. Customer behavioural history -- leakage-safe by construction

In [4]:
from features.transaction_features import build_prior_rule_diversity
from features.entity_features import build_historical_customer_context

history = build_historical_customer_context(rule_df, "Rule")
diversity = build_prior_rule_diversity(rule_df)

rule_with_history = rule_df.copy()
rule_with_history["prior_alert_count"] = history["customer_prior_alert_count"]
rule_with_history["prior_distinct_rule_count"] = diversity["customer_prior_distinct_rule_count"]

repeat_customer = rule_with_history["customer_id"].value_counts().index[0]
example = rule_with_history[rule_with_history["customer_id"] == repeat_customer][
    ["customer_id", "Scan Date (Parsed)", "Rule Name", "prior_alert_count", "prior_distinct_rule_count"]
].sort_values("Scan Date (Parsed)", kind="mergesort")
print(example.to_string(index=False))

# prior_distinct_rule_count must never exceed prior_alert_count (can't have seen
# more distinct rules than total prior alerts)
assert (example["prior_distinct_rule_count"] <= example["prior_alert_count"]).all()
print("\nConfirmed: distinct-rule count never exceeds total prior-alert count for this customer.")

          customer_id      Scan Date (Parsed)                                          Rule Name  prior_alert_count  prior_distinct_rule_count
rule:customer:1517786 2026-06-03 19:07:15.000 23 - NATIONALITY - COUNTRY COMBINATION (SEND TRANS                0.0                        0.0
rule:customer:1517786 2026-06-03 19:07:44.000 23 - NATIONALITY - COUNTRY COMBINATION (SEND TRANS                1.0                        1.0
rule:customer:1517786 2026-06-03 19:07:58.000 23 - NATIONALITY - COUNTRY COMBINATION (SEND TRANS                2.0                        1.0
rule:customer:1517786 2026-06-03 19:09:39.000     96 - MULTIPLE BENEFICIARIES (ONE TO MANY) SEND                3.0                        1.0
rule:customer:1517786 2026-06-03 19:09:55.000     96 - MULTIPLE BENEFICIARIES (ONE TO MANY) SEND                4.0                        2.0
rule:customer:1517786 2026-06-03 19:10:28.000     96 - MULTIPLE BENEFICIARIES (ONE TO MANY) SEND                5.0                        2.0

## 4. Full transaction feature matrix

In [5]:
from features.transaction_features import build_transaction_features

matrix, block_names, artifacts = build_transaction_features(rule_df)

print(f"rows={matrix.shape[0]}, features={matrix.shape[1]}, nnz={matrix.nnz}")
print(f"feature blocks: {block_names}")

dense_check = matrix.toarray()
assert not np.isnan(dense_check).any(), "no NaN permitted in the final matrix"
assert not np.isinf(dense_check).any(), "no inf permitted in the final matrix"
print("PASS: no NaN/inf in final matrix")

rows=2244, features=6119, nnz=122526
feature blocks: ['categorical::Transaction Type Code', 'categorical::Branch Description', 'categorical::Currency Name', 'categorical::Beneficiary Relationship', 'categorical::Purpose', 'categorical::Rule Name', 'nationality::Customer Nationality (Normalized)', 'name_tfidf::Beneficiary Name (Normalized)', 'numeric::Beneficiary Name_missing,Beneficiary Id Number_missing,Beneficiary Relationship_missing,Currency Name_missing,Purpose_missing,customer_prior_alert_count,customer_prior_distinct_rule_count']


PASS: no NaN/inf in final matrix


## 5. Unseen-rule / unseen-category robustness

A rule engine can add a new rule number over time. Scoring an alert with a rule name never seen during training must not crash.

In [6]:
from features.transaction_features import fit_transaction_feature_artifacts, transform_transaction_features

n = len(rule_df)
train_slice = rule_df.iloc[: n // 2]
holdout_slice = rule_df.iloc[n // 2 :].copy()
holdout_slice.iloc[0, holdout_slice.columns.get_loc("Rule Name")] = "999 - A RULE ADDED AFTER TRAINING"

train_artifacts = fit_transaction_feature_artifacts(train_slice)
holdout_matrix, _ = transform_transaction_features(holdout_slice, train_artifacts)
print(f"Transformed holdout containing a never-seen rule name, no error. Shape: {holdout_matrix.shape}")

Transformed holdout containing a never-seen rule name, no error. Shape: (1122, 4840)


## 6. Persist fitted artifacts

In [7]:
from features.transaction_features import save_transaction_feature_artifacts

out_dir = REPO_ROOT / "models" / "transaction"
save_transaction_feature_artifacts(artifacts, out_dir)
manifest_path = out_dir / f"Rule_{artifacts.feature_version}_manifest.json"
manifest = json.load(open(manifest_path))
print(json.dumps(manifest, indent=2, default=str))

{
  "sheet_name": "Rule",
  "feature_version": "transaction-v1",
  "fitted_at": "2026-08-11T06:24:23.220216+00:00",
  "categorical_columns": [
    "Transaction Type Code",
    "Branch Description",
    "Currency Name",
    "Beneficiary Relationship",
    "Purpose",
    "Rule Name"
  ],
  "categorical_category_counts": {
    "Transaction Type Code": 6,
    "Branch Description": 39,
    "Currency Name": 32,
    "Beneficiary Relationship": 51,
    "Purpose": 24,
    "Rule Name": 18
  },
  "nationality_column": "Customer Nationality (Normalized)",
  "beneficiary_name_vocabulary_size": 5893,
  "excluded_features": {
    "transaction_amount": "no clean structured field in current sample; amount only appears in the leakage-typed Comment field, which this module never reads. Requires client-supplied structured field.",
    "customer_currency": "constant in current sample, zero signal.",
    "purpose_code": "redundant with Purpose (same cardinality/coverage)."
  },
  "feature_block_order": [
  

## 7. Phase 4 feature-audit report

In [8]:
feature_audit = {
    "phase": "4_transaction_features",
    "status": "PASS",
    "sheet": "Rule",
    "rows": matrix.shape[0],
    "feature_dims": matrix.shape[1],
    "feature_blocks": block_names,
    "feature_version": artifacts.feature_version,
    "checks": {
        "no_nan_or_inf_in_final_matrix": True,
        "no_leakage_column_used_as_feature_source": True,
        "comment_field_never_read_by_this_module": True,
        "unseen_rule_category_handled_without_error": True,
        "prior_alert_count_and_diversity_are_temporal_safe": True,
        "artifacts_persisted_for_reuse_at_inference": True,
    },
    "excluded_features": manifest["excluded_features"],
    "known_limitations": [
        "No structured transaction amount available in the current sample -- "
        "requires client-supplied structured field before this pipeline can "
        "include amount-based behavioural features (e.g. amount z-score, "
        "amount vs. customer historical average).",
        "customer_prior_alert_count / customer_prior_distinct_rule_count share "
        "the same same-timestamp tie-breaking caveat documented in Phase 3 "
        "(entity_features.build_historical_customer_context docstring).",
    ],
    "next_gate": "Phase 5 -- Entity model experiments (Isolation Forest / Autoencoder benchmarks). Requires human approval before proceeding.",
}

out_path = REPO_ROOT / "evaluation" / "phase4_transaction_features_report.json"
with open(out_path, "w") as f:
    json.dump(feature_audit, f, indent=2, default=str)
print(f"Report written to {out_path}")
print(json.dumps(feature_audit, indent=2, default=str))

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase4_transaction_features_report.json
{
  "phase": "4_transaction_features",
  "status": "PASS",
  "sheet": "Rule",
  "rows": 2244,
  "feature_dims": 6119,
  "feature_blocks": [
    "categorical::Transaction Type Code",
    "categorical::Branch Description",
    "categorical::Currency Name",
    "categorical::Beneficiary Relationship",
    "categorical::Purpose",
    "categorical::Rule Name",
    "nationality::Customer Nationality (Normalized)",
    "name_tfidf::Beneficiary Name (Normalized)",
    "numeric::Beneficiary Name_missing,Beneficiary Id Number_missing,Beneficiary Relationship_missing,Currency Name_missing,Purpose_missing,customer_prior_alert_count,customer_prior_distinct_rule_count"
  ],
  "feature_version": "transaction-v1",
  "checks": {
    "no_nan_or_inf_in_final_matrix": true,
    "no_leakage_column_used_as_feature_source": true,
    "comment_field_never_read_by_this_module": true,
  

## 8. Test suite

In [9]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "tests/", "-q"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 4 is considered done" 

........................................................................ [ 88%]
.........                                                                [100%]
81 passed in 5.07s



## Phase 4 -- Result

**Status: PASS**

- Context categorical features (transaction type, branch, currency, beneficiary relationship, purpose, rule name) -- all grounded in observed cardinality.
- Rule Name used as one shared context feature among 18 values, per master plan: "Do not train 18 independent rule models in the first PoC."
- Beneficiary Name representation via the same char n-gram TF-IDF approach as Phase 3, for the one field the master plan lists as "use when present."
- Customer behavioural history: prior-alert count and prior-*distinct*-rule count, both temporal-safe (strictly-prior only, same tie-breaking caveat as Phase 3, documented not hidden).
- **Transaction amount deliberately excluded** -- no structured field exists, and the only place amount appears (`Comment`) is a leakage field. Verified by a test that scans this module's source and confirms `Comment` is never read as a column, not just documented in a comment.
- Unseen-rule-name robustness confirmed (a rule engine can add new rule numbers over time; scoring must not crash).
- 81/81 tests passing (8 new for this phase).

**Next gate:** Phase 5 -- Entity model experiments. First model training in this project: Isolation Forest (starting candidate, not pre-declared winner) and Autoencoder benchmarked under the same leakage-controlled, group+time validation protocol, using the Phase 3 entity representations. Awaiting human approval to proceed.